# 🏌️ Double Pendulum — Golf Swing Dynamics

Interactive simulation of a **driven double pendulum** modelled as a simplified golf swing.
Uses **Lagrangian mechanics with relative (generalised) coordinates**.

---

### What this demonstrates

| Term | Physical meaning |
|------|------------------|
| **M₁₁, M₂₂** | Self-inertia — each joint's resistance to its own acceleration |
| **M₁₂ = M₂₁** | Cross-coupling — how shoulder torque drives club acceleration *without* wrist torque |
| **Friction** | Combined viscous damping (b·ω) and Coulomb friction (μ·sign(ω)) |

### Coordinates
```
Shoulder (fixed pivot)
    │ segment 1 (arm): θ₁ from vertical
Wrist (joint)
    │ segment 2 (club): φ relative to arm
Club tip
```

In [ ]:
# ── Install dependencies (Colab/standalone) ──────────────────────────────────
import sys
try:
    import scipy
    import ipywidgets
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'scipy', 'ipywidgets', 'matplotlib'])

print('✅ Dependencies ready')

In [ ]:
"""
Physics Engine — Double Pendulum (Lagrangian, relative coordinates).

Design by Contract:
    Pre-conditions are enforced with assert statements.
    Post-conditions are asserted on critically derived quantities.
All functions are pure — no global state.
"""
from __future__ import annotations
from dataclasses import dataclass
from typing import Callable, Tuple
import numpy as np
from scipy.integrate import solve_ivp


# ── Data structures ───────────────────────────────────────────────────────────

@dataclass(frozen=True)
class PendulumParams:
    """Immutable physical parameters.

    DbC:
        - m1, m2, L1, L2 > 0
        - g >= 0
        - b1, b2, mu1, mu2 >= 0  (dissipative, cannot add energy)
    """
    m1: float  # kg  — segment 1 (arm)
    m2: float  # kg  — segment 2 (club)
    L1: float  # m   — segment 1 length
    L2: float  # m   — segment 2 length
    g:  float = 9.81   # m/s²
    b1: float = 0.0   # viscous damping joint 1 [N·m·s/rad]
    b2: float = 0.0   # viscous damping joint 2 [N·m·s/rad]
    mu1: float = 0.0  # Coulomb friction joint 1 [N·m]
    mu2: float = 0.0  # Coulomb friction joint 2 [N·m]

    def __post_init__(self) -> None:
        assert self.m1 > 0,   f'm1 must be positive, got {self.m1}'
        assert self.m2 > 0,   f'm2 must be positive, got {self.m2}'
        assert self.L1 > 0,   f'L1 must be positive, got {self.L1}'
        assert self.L2 > 0,   f'L2 must be positive, got {self.L2}'
        assert self.g  >= 0,  f'g must be non-negative, got {self.g}'
        assert self.b1 >= 0,  f'b1 must be non-negative, got {self.b1}'
        assert self.b2 >= 0,  f'b2 must be non-negative, got {self.b2}'
        assert self.mu1 >= 0, f'mu1 must be non-negative, got {self.mu1}'
        assert self.mu2 >= 0, f'mu2 must be non-negative, got {self.mu2}'


State = np.ndarray  # [theta1, phi, dtheta1, dphi]
TorqueFunc = Callable[[float], Tuple[float, float]]


# ── Mass matrix ───────────────────────────────────────────────────────────────

def mass_matrix(phi: float, p: PendulumParams) -> np.ndarray:
    """2×2 symmetric positive-definite inertia matrix.

    Pre:  phi is finite.
    Post: M is symmetric, M[1,1] > 0.
    """
    assert np.isfinite(phi), f'phi must be finite, got {phi}'
    c = np.cos(phi)
    M11 = (p.m1 + p.m2)*p.L1**2 + p.m2*p.L2**2 + 2*p.m2*p.L1*p.L2*c
    M12 = p.m2*p.L2**2 + p.m2*p.L1*p.L2*c
    M22 = p.m2*p.L2**2
    M = np.array([[M11, M12], [M12, M22]])
    assert np.isclose(M[0,1], M[1,0]), 'Mass matrix must be symmetric'
    assert M[1,1] > 0, 'M22 must be positive'
    return M

def mass_matrix_components(phi: float, p: PendulumParams) -> dict:
    M = mass_matrix(phi, p)
    return {'M11': M[0,0], 'M12': M[0,1], 'M21': M[1,0], 'M22': M[1,1], 'M_full': M}


# ── Coriolis / centrifugal ────────────────────────────────────────────────────

def coriolis_vector(phi: float, dtheta1: float, dphi: float, p: PendulumParams) -> np.ndarray:
    """Velocity-dependent forces C(q,q̇)·q̇."""
    assert all(np.isfinite(v) for v in [phi, dtheta1, dphi]), 'Velocities must be finite'
    h = -p.m2 * p.L1 * p.L2 * np.sin(phi)
    return np.array([h*(2*dtheta1*dphi + dphi**2), -h*dtheta1**2])


# ── Gravity ───────────────────────────────────────────────────────────────────

def gravity_vector(theta1: float, phi: float, p: PendulumParams) -> np.ndarray:
    """Gravitational torque vector G(q)."""
    a2 = theta1 + phi
    G1 = (p.m1+p.m2)*p.g*p.L1*np.sin(theta1) + p.m2*p.g*p.L2*np.sin(a2)
    G2 = p.m2*p.g*p.L2*np.sin(a2)
    return np.array([G1, G2])


# ── Friction ──────────────────────────────────────────────────────────────────

def friction_torque_vector(dtheta1: float, dphi: float, p: PendulumParams) -> np.ndarray:
    """Dissipative joint torques: viscous + Coulomb, opposing motion.

    Pre:  dtheta1, dphi finite.
    Post: each component opposes the corresponding velocity (or zero).
    """
    assert np.isfinite(dtheta1) and np.isfinite(dphi), 'Velocities must be finite'
    tau_f1 = -p.b1*dtheta1 - p.mu1*np.sign(dtheta1)
    tau_f2 = -p.b2*dphi   - p.mu2*np.sign(dphi)
    return np.array([tau_f1, tau_f2])


# ── Equations of motion ───────────────────────────────────────────────────────

def equations_of_motion(state: State, t: float, p: PendulumParams,
                         torque_func: TorqueFunc) -> State:
    """M(q)·q̈ = τ_drive(t) + τ_friction(q̇) - C(q,q̇) - G(q).

    Pre:  state shape (4,), all finite.
    Post: state_dot shape (4,), all finite.
    """
    assert state.shape == (4,) and all(np.isfinite(state)), 'Invalid state'
    theta1, phi, dtheta1, dphi = state
    M = mass_matrix(phi, p)
    C = coriolis_vector(phi, dtheta1, dphi, p)
    G = gravity_vector(theta1, phi, p)
    tau1, tau2 = torque_func(t)
    tau_drive    = np.array([tau1, tau2])
    tau_friction = friction_torque_vector(dtheta1, dphi, p)
    rhs = tau_drive + tau_friction - C - G
    qddot = np.linalg.solve(M, rhs)
    state_dot = np.array([dtheta1, dphi, qddot[0], qddot[1]])
    assert all(np.isfinite(state_dot)), f'Non-finite state_dot: {state_dot}'
    return state_dot


# ── Forward kinematics ────────────────────────────────────────────────────────

def forward_kinematics(theta1: float, phi: float, p: PendulumParams) -> dict:
    """Joint positions in world frame (shoulder at origin)."""
    a2 = theta1 + phi
    wx, wy = p.L1*np.sin(theta1), -p.L1*np.cos(theta1)
    tx = wx + p.L2*np.sin(a2)
    ty = wy - p.L2*np.cos(a2)
    return {'shoulder': (0.0, 0.0), 'wrist': (wx, wy), 'tip': (tx, ty)}


# ── Energy ────────────────────────────────────────────────────────────────────

def kinetic_energy(state: State, p: PendulumParams) -> float:
    _, phi, dtheta1, dphi = state
    M = mass_matrix(phi, p)
    qdot = np.array([dtheta1, dphi])
    return 0.5 * qdot @ M @ qdot

def potential_energy(state: State, p: PendulumParams) -> float:
    theta1, phi = state[0], state[1]
    a2 = theta1 + phi
    return -(p.m1+p.m2)*p.g*p.L1*np.cos(theta1) - p.m2*p.g*p.L2*np.cos(a2)

def total_energy(state: State, p: PendulumParams) -> float:
    return kinetic_energy(state, p) + potential_energy(state, p)


# ── Polynomial torque builder ─────────────────────────────────────────────────

def make_polynomial_torque(coeffs_shoulder: list, coeffs_wrist: list) -> TorqueFunc:
    """τ(t) = c₀ + c₁t + c₂t² + …

    Pre: each coefficient list has ≥ 1 element.
    """
    assert len(coeffs_shoulder) >= 1 and len(coeffs_wrist) >= 1
    p_sh = np.array(coeffs_shoulder[::-1])
    p_wr = np.array(coeffs_wrist[::-1])
    def _torque(t: float) -> Tuple[float, float]:
        return float(np.polyval(p_sh, t)), float(np.polyval(p_wr, t))
    return _torque


# ── Simulation runner ─────────────────────────────────────────────────────────

def run_simulation(params:PendulumParams, initial_state:State, t_end:float,
                   torque_func:TorqueFunc, dt:float=0.005,
                   method:str='RK45') -> dict:
    """Integrate EOM with scipy solve_ivp.

    Returns dict with keys: t, states, params, torque_func.

    Pre:
        - initial_state shape (4,), all finite
        - t_end > 0, 0 < dt < t_end
    Post:
        - len(result['t']) >= 2, all state values finite
    """
    assert initial_state.shape == (4,) and all(np.isfinite(initial_state))
    assert t_end > 0 and 0 < dt < t_end

    t_eval = np.arange(0.0, t_end, dt)

    sol = solve_ivp(
        lambda t, y: equations_of_motion(y, t, params, torque_func),
        t_span=(0.0, t_end),
        y0=initial_state,
        t_eval=t_eval,
        method=method,
        rtol=1e-8, atol=1e-10,
        max_step=dt,
    )
    if not sol.success:
        raise RuntimeError(f'Integration failed: {sol.message}')

    result = {'t': sol.t, 'states': sol.y.T, 'params': params, 'torque_func': torque_func}
    assert len(result['t']) >= 2, 'Simulation must produce ≥ 2 timesteps'
    return result


print('✅ Physics engine loaded')

In [ ]:
# ── Built-in presets (DRY: shared with GUI) ───────────────────────────────────

PRESETS = {
    'Golf Swing (passive wrist)': dict(
        m1=5.0, m2=0.3, L1=0.65, L2=1.1,
        b1=0.1, b2=0.05, mu1=0.02, mu2=0.01,
        theta1_deg=-60.0, phi_deg=80.0,
        dtheta1=0.0, dphi=0.0,
        coeffs_shoulder=[-25.0, 10.0],
        coeffs_wrist=[0.0],
        t_end=2.0,
    ),
    'Golf Swing (active wrist)': dict(
        m1=5.0, m2=0.3, L1=0.65, L2=1.1,
        b1=0.1, b2=0.05, mu1=0.02, mu2=0.01,
        theta1_deg=-60.0, phi_deg=80.0,
        dtheta1=0.0, dphi=0.0,
        coeffs_shoulder=[-25.0, 10.0],
        coeffs_wrist=[5.0, -3.0],
        t_end=2.0,
    ),
    'Free Double Pendulum': dict(
        m1=1.0, m2=1.0, L1=1.0, L2=1.0,
        b1=0.0, b2=0.0, mu1=0.0, mu2=0.0,
        theta1_deg=90.0, phi_deg=0.0,
        dtheta1=0.0, dphi=0.0,
        coeffs_shoulder=[0.0],
        coeffs_wrist=[0.0],
        t_end=5.0,
    ),
    'Straight Drop': dict(
        m1=2.0, m2=1.0, L1=0.8, L2=0.8,
        b1=0.0, b2=0.0, mu1=0.0, mu2=0.0,
        theta1_deg=5.0, phi_deg=0.0,
        dtheta1=0.0, dphi=0.0,
        coeffs_shoulder=[0.0],
        coeffs_wrist=[0.0],
        t_end=3.0,
    ),
}

print(f'✅ {len(PRESETS)} presets available:', list(PRESETS.keys()))

In [ ]:
# ── Run a simulation from a preset ───────────────────────────────────────────
import math

def run_preset(name: str) -> dict:
    """Run simulation from a named preset and return result dict."""
    assert name in PRESETS, f'Unknown preset: {name!r}'
    p = PRESETS[name]
    params = PendulumParams(
        m1=p['m1'], m2=p['m2'], L1=p['L1'], L2=p['L2'],
        b1=p['b1'], b2=p['b2'], mu1=p['mu1'], mu2=p['mu2'],
    )
    init_state = np.array([
        math.radians(p['theta1_deg']),
        math.radians(p['phi_deg']),
        p['dtheta1'], p['dphi'],
    ])
    torque_func = make_polynomial_torque(p['coeffs_shoulder'], p['coeffs_wrist'])
    result = run_simulation(params, init_state, p['t_end'], torque_func)
    result['preset_name'] = name
    return result


# Run the default preset
result = run_preset('Golf Swing (passive wrist)')
print(f"✅ Simulated {result['preset_name']}: {len(result['t'])} time steps,",
      f"t = {result['t'][0]:.3f} … {result['t'][-1]:.3f} s")

In [ ]:
# ── Static analysis plots ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('dark_background')

COLORS = dict(theta1='#60aaff', phi='#ff9955', dtheta1='#44ddaa',
              dphi='#ff55aa', KE='#60aaff', PE='#ff9955', E='#ffffff',
              tau1='#60aaff', tau2='#ff9955', tau_total='#44ddaa')


def plot_full_analysis(result: dict, figsize=(16, 12)) -> None:
    """Four-panel analysis: angles, velocities, energy, and total torques."""
    t = result['t']
    states = result['states']
    p = result['params']
    tf = result['torque_func']
    name = result.get('preset_name', 'Simulation')

    theta1 = np.degrees(states[:, 0])
    phi    = np.degrees(states[:, 1])
    dth1   = np.degrees(states[:, 2])
    dphi   = np.degrees(states[:, 3])

    # Energy along trajectory
    KE_arr = np.array([kinetic_energy(states[i], p) for i in range(len(t))])
    PE_arr = np.array([potential_energy(states[i], p) for i in range(len(t))])
    E_arr  = KE_arr + PE_arr

    # Total torques (drive + friction)
    tau1_drive = np.array([tf(ti)[0] for ti in t])
    tau2_drive = np.array([tf(ti)[1] for ti in t])
    tau1_fric  = np.array([-p.b1*states[i,2] - p.mu1*np.sign(states[i,2]) for i in range(len(t))])
    tau2_fric  = np.array([-p.b2*states[i,3] - p.mu2*np.sign(states[i,3]) for i in range(len(t))])
    tau1_total = tau1_drive + tau1_fric
    tau2_total = tau2_drive + tau2_fric

    fig = plt.figure(figsize=figsize, facecolor='#121218')
    gs  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)
    axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]

    def _style(ax, title, xlabel='Time (s)', ylabel=''):
        ax.set_facecolor('#1a1a28')
        ax.set_title(title, color='#c0c0e0', fontsize=11, pad=6)
        ax.set_xlabel(xlabel, color='#8080a0', fontsize=9)
        ax.set_ylabel(ylabel, color='#8080a0', fontsize=9)
        ax.tick_params(colors='#6060a0', labelsize=8)
        ax.spines[:].set_color('#303048')
        ax.grid(True, alpha=0.18, color='#4040a0')
        ax.legend(fontsize=8, facecolor='#1a1a28', edgecolor='#303048', labelcolor='white')

    # 1 — Angles
    axes[0].plot(t, theta1, color=COLORS['theta1'], lw=1.5, label='θ₁ (arm)')
    axes[0].plot(t, phi,    color=COLORS['phi'],    lw=1.5, label='φ (club)')
    _style(axes[0], 'Joint Angles (°)')

    # 2 — Angular velocities
    axes[1].plot(t, dth1, color=COLORS['dtheta1'], lw=1.5, label='θ̇₁')
    axes[1].plot(t, dphi, color=COLORS['dphi'],    lw=1.5, label='φ̇')
    _style(axes[1], 'Angular Velocities (°/s)')

    # 3 — Energy
    axes[2].plot(t, KE_arr, color=COLORS['KE'], lw=1.5, label='KE')
    axes[2].plot(t, PE_arr, color=COLORS['PE'], lw=1.5, label='PE')
    axes[2].plot(t, E_arr,  color=COLORS['E'],  lw=2.0, label='Total E', ls='--')
    _style(axes[2], 'Energy (J)', ylabel='J')

    # 4 — Total applied torques (drive + friction)
    axes[3].plot(t, tau1_total, color=COLORS['tau1'], lw=1.5, label='τ_total shoulder')
    axes[3].plot(t, tau2_total, color=COLORS['tau2'], lw=1.5, label='τ_total wrist')
    axes[3].fill_between(t, tau1_drive, tau1_total, alpha=0.2, color=COLORS['tau1'],
                         label='Friction component (shoulder)')
    axes[3].fill_between(t, tau2_drive, tau2_total, alpha=0.2, color=COLORS['tau2'],
                         label='Friction component (wrist)')
    _style(axes[3], 'Total Applied Torques (N·m)', ylabel='N·m')

    fig.suptitle(f'Double Pendulum — {name}', color='#d0d0e8', fontsize=14, y=1.01)
    plt.show()


plot_full_analysis(result)

In [ ]:
# ── Animation (skip on Colab if no JS kernel) ─────────────────────────────────
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def animate_pendulum(result: dict, fps: int = 40, trail_len: int = 80) -> HTML:
    """Return an HTML5 video animation of the pendulum trajectory."""
    t      = result['t']
    states = result['states']
    p      = result['params']
    name   = result.get('preset_name', '')
    stride = max(1, int(round(1.0 / (fps * (t[1]-t[0])))))

    fig, ax = plt.subplots(1, 1, figsize=(5, 6), facecolor='#121218')
    ax.set_facecolor('#1a1a28')
    ax.set_xlim(-p.L1-p.L2-0.1, p.L1+p.L2+0.1)
    ax.set_ylim(-p.L1-p.L2-0.1, p.L1+0.1)
    ax.set_aspect('equal')
    ax.spines[:].set_color('#303048')
    ax.tick_params(colors='#6060a0', labelsize=7)
    ax.set_title(name, color='#c0c0e0', fontsize=10)

    line,   = ax.plot([], [], 'o-', lw=2.5, color='#70ccff', ms=8)
    trail,  = ax.plot([], [], '-', lw=1.0, color='#ff9955', alpha=0.5)
    time_tx  = ax.text(0.02, 0.96, '', transform=ax.transAxes,
                       color='#c0c0e0', fontsize=9)

    tip_xs, tip_ys = [], []

    def _init():
        line.set_data([], [])
        trail.set_data([], [])
        return line, trail, time_tx

    def _update(frame):
        i = frame * stride
        if i >= len(t): i = len(t)-1
        pos = forward_kinematics(states[i,0], states[i,1], p)
        xs = [pos['shoulder'][0], pos['wrist'][0], pos['tip'][0]]
        ys = [pos['shoulder'][1], pos['wrist'][1], pos['tip'][1]]
        line.set_data(xs, ys)
        tip_xs.append(pos['tip'][0])
        tip_ys.append(pos['tip'][1])
        tr = tip_xs[-trail_len:]
        trail.set_data(tip_xs[-trail_len:], tip_ys[-trail_len:])
        time_tx.set_text(f't = {t[i]:.2f} s')
        return line, trail, time_tx

    n_frames = len(t) // stride
    anim = FuncAnimation(fig, _update, frames=n_frames, init_func=_init,
                         blit=True, interval=1000//fps)
    plt.close(fig)
    return HTML(anim.to_jshtml())


animate_pendulum(result)

In [ ]:
# ── Interactive ipywidgets panel ──────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output

out = widgets.Output()

preset_dd = widgets.Dropdown(
    options=list(PRESETS.keys()),
    value='Golf Swing (passive wrist)',
    description='Preset:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='360px'),
)

# Physical parameters
sliders = {
    'm1':  widgets.FloatSlider(min=0.5, max=10.0, step=0.1, value=5.0,  description='m₁ (kg)',    style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    'm2':  widgets.FloatSlider(min=0.05,max=2.0,  step=0.05,value=0.3,  description='m₂ (kg)',    style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    'L1':  widgets.FloatSlider(min=0.2, max=2.0,  step=0.05,value=0.65, description='L₁ (m)',     style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    'L2':  widgets.FloatSlider(min=0.2, max=2.0,  step=0.05,value=1.1,  description='L₂ (m)',     style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    'b1':  widgets.FloatSlider(min=0.0, max=2.0,  step=0.01,value=0.1,  description='b₁ damp',    style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    'b2':  widgets.FloatSlider(min=0.0, max=2.0,  step=0.01,value=0.05, description='b₂ damp',    style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    'mu1': widgets.FloatSlider(min=0.0, max=1.0,  step=0.005,value=0.02,description='μ₁ Coulomb', style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    'mu2': widgets.FloatSlider(min=0.0, max=1.0,  step=0.005,value=0.01,description='μ₂ Coulomb', style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    'th1': widgets.FloatSlider(min=-180,max=180,  step=1.0, value=-60.0,description='θ₁₀ (°)',    style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    'phi': widgets.FloatSlider(min=-180,max=180,  step=1.0, value=80.0, description='φ₀ (°)',     style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
    't_end': widgets.FloatSlider(min=0.5,max=10.0,step=0.5, value=2.0,  description='Duration (s)',style={'description_width':'80px'}, layout=widgets.Layout(width='400px')),
}

run_btn = widgets.Button(description='▶ Run Simulation', button_style='success',
                         layout=widgets.Layout(width='200px', height='36px'))

anim_btn = widgets.Button(description='🎬 Animate', button_style='info',
                          layout=widgets.Layout(width='150px', height='36px'))

_last_result = {}


def _load_preset(change):
    p = PRESETS[preset_dd.value]
    sliders['m1'].value   = p['m1']
    sliders['m2'].value   = p['m2']
    sliders['L1'].value   = p['L1']
    sliders['L2'].value   = p['L2']
    sliders['b1'].value   = p['b1']
    sliders['b2'].value   = p['b2']
    sliders['mu1'].value  = p['mu1']
    sliders['mu2'].value  = p['mu2']
    sliders['th1'].value  = p['theta1_deg']
    sliders['phi'].value  = p['phi_deg']
    sliders['t_end'].value = p['t_end']

preset_dd.observe(_load_preset, names='value')


def _run(_):
    global _last_result
    with out:
        clear_output(wait=True)
        try:
            params = PendulumParams(
                m1=sliders['m1'].value, m2=sliders['m2'].value,
                L1=sliders['L1'].value, L2=sliders['L2'].value,
                b1=sliders['b1'].value, b2=sliders['b2'].value,
                mu1=sliders['mu1'].value, mu2=sliders['mu2'].value,
            )
            p = PRESETS[preset_dd.value]
            init = np.array([
                math.radians(sliders['th1'].value),
                math.radians(sliders['phi'].value),
                0.0, 0.0,
            ])
            torque_func = make_polynomial_torque(p['coeffs_shoulder'], p['coeffs_wrist'])
            _last_result = run_simulation(params, init, sliders['t_end'].value, torque_func)
            _last_result['preset_name'] = preset_dd.value
            plot_full_analysis(_last_result)
        except Exception as exc:
            print(f'❌ Error: {exc}')


def _animate(_):
    if not _last_result:
        print('Run a simulation first.')
        return
    with out:
        clear_output(wait=True)
        display(animate_pendulum(_last_result))


run_btn.on_click(_run)
anim_btn.on_click(_animate)

col1 = widgets.VBox([sliders['m1'], sliders['m2'], sliders['L1'], sliders['L2'],
                     sliders['t_end']])
col2 = widgets.VBox([sliders['b1'], sliders['b2'], sliders['mu1'], sliders['mu2'],
                     sliders['th1'], sliders['phi']])

ui = widgets.VBox([
    widgets.HTML('<h3 style="color:#c0d0ff">🏌️ Double Pendulum Golf Swing Simulator</h3>'),
    preset_dd,
    widgets.HBox([col1, col2]),
    widgets.HBox([run_btn, anim_btn]),
    out,
])

display(ui)

## ✅ TDD — Inline Test Suite

Run the cell below to execute the contract / physics tests in-notebook.
These mirror the full pytest suite; each test uses `assert` for clarity.

In [ ]:
# ── Inline TDD tests (DbC + physics invariants) ───────────────────────────────
import traceback

_PASS = '✅'
_FAIL = '❌'

def _run_test(name, fn):
    try:
        fn()
        print(f'{_PASS} {name}')
        return True
    except Exception as e:
        print(f'{_FAIL} {name}: {e}')
        traceback.print_exc()
        return False

_params_std = PendulumParams(m1=1.0, m2=1.0, L1=1.0, L2=1.0)
_params_damp = PendulumParams(m1=1.0, m2=1.0, L1=1.0, L2=1.0, b1=0.5, b2=0.3, mu1=0.1, mu2=0.05)

tests = [
    # DbC: invalid params rejected
    ('DbC: negative mass rejected', lambda: [
        (lambda: PendulumParams(m1=-1.0, m2=1.0, L1=1.0, L2=1.0))(),
        (_ for _ in ()).throw(AssertionError),
    ] if False else _assert_raises(AssertionError, PendulumParams, m1=-1.0, m2=1.0, L1=1.0, L2=1.0)),
]

def _assert_raises(exc_type, fn, **kwargs):
    try:
        fn(**kwargs)
        raise AssertionError(f'Expected {exc_type.__name__} but no exception raised')
    except exc_type:
        pass

def test_negative_mass():
    _assert_raises(AssertionError, PendulumParams, m1=-1.0, m2=1.0, L1=1.0, L2=1.0)

def test_zero_length():
    _assert_raises(AssertionError, PendulumParams, m1=1.0, m2=1.0, L1=0.0, L2=1.0)

def test_mass_matrix_symmetric():
    M = mass_matrix(0.5, _params_std)
    assert np.isclose(M[0,1], M[1,0]), 'Not symmetric'

def test_mass_matrix_positive_definite():
    for phi in np.linspace(-np.pi, np.pi, 20):
        M = mass_matrix(phi, _params_std)
        eigenvalues = np.linalg.eigvalsh(M)
        assert all(ev > 0 for ev in eigenvalues), f'Not PD at phi={phi:.2f}'

def test_equilibrium_no_acceleration():
    """At rest in equilibrium (theta1=phi=0, zero velocity, no torque) accel should be ~0."""
    state = np.array([0.0, 0.0, 0.0, 0.0])
    zero_torque = lambda t: (0.0, 0.0)
    sd = equations_of_motion(state, 0.0, _params_std, zero_torque)
    assert np.allclose(sd[2:], 0.0, atol=1e-10), f'Non-zero accel at equilibrium: {sd[2:]}'

def test_energy_conserved_no_friction_no_torque():
    """With no friction/torque, |ΔE| < 1e-4 over 2 s."""
    state0 = np.array([0.3, 0.1, 0.0, 0.0])
    res = run_simulation(_params_std, state0, 2.0, lambda t: (0.0, 0.0))
    E0 = total_energy(res['states'][0],  _params_std)
    Ef = total_energy(res['states'][-1], _params_std)
    assert abs(Ef - E0) < 1e-4, f'|ΔE| = {abs(Ef-E0):.2e} (> 1e-4)'

def test_friction_reduces_energy():
    """With damping and no driving torque, energy must strictly decrease."""
    state0 = np.array([0.5, 0.2, 0.5, 0.3])
    res = run_simulation(_params_damp, state0, 3.0, lambda t: (0.0, 0.0))
    E0 = total_energy(res['states'][0],  _params_damp)
    Ef = total_energy(res['states'][-1], _params_damp)
    assert Ef < E0, f'Energy not reduced by damping: E0={E0:.4f}, Ef={Ef:.4f}'

def test_friction_torque_opposes_motion():
    """Friction torques must oppose positive velocity (negative tau)."""
    tau_f = friction_torque_vector(1.0, 2.0, _params_damp)
    assert tau_f[0] < 0, 'Friction at joint 1 must be negative for positive velocity'
    assert tau_f[1] < 0, 'Friction at joint 2 must be negative for positive velocity'

def test_polynomial_torque_constant():
    tf = make_polynomial_torque([5.0], [0.0])
    for t in [0.0, 0.5, 1.0, 2.0]:
        tau1, tau2 = tf(t)
        assert abs(tau1 - 5.0) < 1e-12, f'tau1={tau1} ≠ 5.0 at t={t}'
        assert abs(tau2) < 1e-12, f'tau2={tau2} ≠ 0.0 at t={t}'

def test_forward_kinematics_hanging():
    """When theta1=phi=0, wrist at (0,-L1), tip at (0,-L1-L2)."""
    pos = forward_kinematics(0.0, 0.0, _params_std)
    assert np.isclose(pos['wrist'][0], 0.0) and np.isclose(pos['wrist'][1], -1.0)
    assert np.isclose(pos['tip'][0],   0.0) and np.isclose(pos['tip'][1],   -2.0)

all_tests = [
    ('DbC: negative mass rejected', test_negative_mass),
    ('DbC: zero length rejected',   test_zero_length),
    ('Mass matrix symmetric',       test_mass_matrix_symmetric),
    ('Mass matrix positive-definite', test_mass_matrix_positive_definite),
    ('Equilibrium → zero accel',    test_equilibrium_no_acceleration),
    ('Energy conserved (no friction, no torque)', test_energy_conserved_no_friction_no_torque),
    ('Friction reduces energy',     test_friction_reduces_energy),
    ('Friction opposes motion',     test_friction_torque_opposes_motion),
    ('Polynomial torque (constant)', test_polynomial_torque_constant),
    ('Forward kinematics (hanging)', test_forward_kinematics_hanging),
]

passed = sum(_run_test(name, fn) for name, fn in all_tests)
print(f'\n{passed}/{len(all_tests)} tests passed.')